# EDA rapido - vehicles_us.csv

## 1. Carga de datos

In [10]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../vehicles_us.csv')
df.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,23/06/2018,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,19/10/2018,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,07/02/2019,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,22/03/2019,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,02/04/2019,28


## 2. Exploracion basica

In [11]:
print('shape:', df.shape)
print('\ncolumnas:')
print(df.columns.tolist())
print('\ndtypes:')
print(df.dtypes)
print('\ninfo():')
df.info()
print('\nisna().sum():')
print(df.isna().sum())
print('\nduplicated().sum():', df.duplicated().sum())

shape: (51525, 13)

columnas:
['price', 'model_year', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'transmission', 'type', 'paint_color', 'is_4wd', 'date_posted', 'days_listed']

dtypes:
price             int64
model_year      float64
model            object
condition        object
cylinders       float64
fuel             object
odometer        float64
transmission     object
type             object
paint_color      object
is_4wd          float64
date_posted      object
days_listed       int64
dtype: object

info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  object 
 3   condition     51525 non-null  object 
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  object 
 6   odometer  

## 3. Limpieza minima

In [12]:
target_columns = ['days_listed', 'type', 'model', 'price']

rows_before = df.shape[0]
duplicates_before = df.duplicated().sum()
nulls_before = df[target_columns].isna().sum()

df_clean = df.drop_duplicates().copy()
df_clean = df_clean.dropna(subset=target_columns)

if not pd.api.types.is_numeric_dtype(df_clean['days_listed']):
    df_clean['days_listed'] = pd.to_numeric(df_clean['days_listed'], errors='coerce')
if not pd.api.types.is_numeric_dtype(df_clean['price']):
    df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')

df_clean = df_clean.dropna(subset=target_columns)
df_clean['inventory_days'] = df_clean['days_listed']

rows_after = df_clean.shape[0]
duplicates_after = df_clean.duplicated().sum()
nulls_after = df_clean[target_columns].isna().sum()

print('Resumen de cambios:')
print(f'- Filas antes: {rows_before}')
print(f'- Filas despues: {rows_after}')
print(f'- Duplicados eliminados: {duplicates_before - duplicates_after}')
print('\nNulos antes (columnas objetivo):')
print(nulls_before)
print('\nNulos despues (columnas objetivo):')
print(nulls_after)
print('\nDataset listo para visualizacion basica:', df_clean.shape)
df_clean[target_columns + ['inventory_days']].head()

Resumen de cambios:
- Filas antes: 51525
- Filas despues: 51525
- Duplicados eliminados: 0

Nulos antes (columnas objetivo):
days_listed    0
type           0
model          0
price          0
dtype: int64

Nulos despues (columnas objetivo):
days_listed    0
type           0
model          0
price          0
dtype: int64

Dataset listo para visualizacion basica: (51525, 14)


,days_listed,type,model,price,inventory_days
0,19,SUV,bmw x5,9400,19
1,50,pickup,ford f-150,25500,50
2,79,sedan,hyundai sonata,5500,79
3,9,pickup,ford f-150,1500,9
4,28,sedan,chrysler 200,14900,28


## 4. Visualizaciones necesarias (plotly.graph_objects)

In [13]:
fig_days_distribution = go.Figure()
fig_days_distribution.add_trace(
    go.Histogram(x=df_clean['days_listed'], nbinsx=40, name='days_listed')
)
fig_days_distribution.update_layout(
    title='Distribucion de days_listed',
    xaxis_title='days_listed',
    yaxis_title='Frecuencia'
)
fig_days_distribution.show()

In [14]:
type_count_df = (
    df_clean.groupby('type', as_index=False)['model']
    .count()
    .rename(columns={'model': 'listing_count'})
    .sort_values('listing_count', ascending=False)
)

fig_type_count = go.Figure()
fig_type_count.add_trace(
    go.Histogram(x=df_clean['type'], name='Conteo por type')
)
fig_type_count.update_layout(
    title='Conteo por type',
    xaxis_title='type',
    yaxis_title='Frecuencia'
)
fig_type_count.show()

type_count_df.head()

,type,listing_count
0,SUV,12405
10,truck,12353
9,sedan,12154
8,pickup,6988
3,coupe,2303


In [15]:
type_days_df = (
    df_clean.groupby('type', as_index=False)['days_listed']
    .mean()
    .sort_values('days_listed', ascending=False)
)

fig_type_vs_days = go.Figure()
fig_type_vs_days.add_trace(
    go.Scatter(
        x=type_days_df['type'],
        y=type_days_df['days_listed'],
        mode='markers',
        name='Promedio days_listed'
    )
)
fig_type_vs_days.update_layout(
    title='type vs days_listed (promedio)',
    xaxis_title='type',
    yaxis_title='days_listed promedio'
)
fig_type_vs_days.show()

In [16]:
model_days_df = (
    df_clean.groupby('model', as_index=False)['days_listed']
    .mean()
    .sort_values('days_listed', ascending=False)
)

top_models_df = (
    df_clean.groupby('model', as_index=False)
    .size()
    .rename(columns={'size': 'listing_count'})
    .sort_values('listing_count', ascending=False)
    .head(20)
)

model_days_top_df = top_models_df.merge(model_days_df, on='model', how='left')

fig_model_vs_days = go.Figure()
fig_model_vs_days.add_trace(
    go.Scatter(
        x=model_days_top_df['model'],
        y=model_days_top_df['days_listed'],
        mode='markers',
        name='Promedio days_listed'
    )
)
fig_model_vs_days.update_layout(
    title='model vs days_listed (top 20 modelos por cantidad)',
    xaxis_title='model',
    yaxis_title='days_listed promedio'
)
fig_model_vs_days.show()

In [17]:
top_types = type_count_df['type'].head(6).tolist()

fig_type_vs_price = go.Figure()
for car_type in top_types:
    type_prices = df_clean.loc[df_clean['type'] == car_type, 'price']
    fig_type_vs_price.add_trace(
        go.Histogram(x=type_prices, name=car_type, opacity=0.6)
    )

fig_type_vs_price.update_layout(
    title='type vs price (top 6 tipos)',
    xaxis_title='price',
    yaxis_title='Frecuencia',
    barmode='overlay'
)
fig_type_vs_price.show()

In [18]:
fig_price_vs_days = go.Figure()
fig_price_vs_days.add_trace(
    go.Scatter(
        x=df_clean['price'],
        y=df_clean['days_listed'],
        mode='markers',
        name='price vs days_listed'
    )
)
fig_price_vs_days.update_layout(
    title='price vs days_listed',
    xaxis_title='price',
    yaxis_title='days_listed'
)
fig_price_vs_days.show()